# Week 3 — Training the CNN Model

**Goal:** Convert the dataset into model-ready arrays, train the CNN, visualize training progress, and save the trained model.

This week turns the model from Week 2 into a working AI system by letting it learn from labeled chest X-ray images.

> Educational note: Training results can vary between runs. This notebook is for learning AI workflow, not for medical diagnosis.

## 1. Install and Import Required Packages

In [ ]:
# Install MedMNIST for access to PneumoniaMNIST.
!pip install medmnist

# matplotlib is used for plotting training graphs.
import matplotlib.pyplot as plt

# NumPy is used for array conversion and normalization.
import numpy as np

# TensorFlow/Keras is used to build and train the CNN.
import tensorflow as tf
from tensorflow.keras import layers, models

# PneumoniaMNIST is the chest X-ray dataset used in this workshop.
from medmnist import PneumoniaMNIST

## 2. Load the Training Dataset

In [ ]:
# Load the training split.
# The model will learn from these labeled examples.
train_dataset = PneumoniaMNIST(split='train', download=True)

# Print the number of training images.
print("Number of training images:", len(train_dataset))

## 3. Build the CNN Model

This is the same basic CNN architecture introduced in Week 2.  
It receives 28×28 grayscale images and outputs one probability.

In [ ]:
# Create a CNN model by stacking layers in order.
model = models.Sequential([
    # Input layer for 28x28 grayscale X-ray images.
    layers.Input(shape=(28, 28, 1)),

    # First feature extraction layer.
    layers.Conv2D(16, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Second feature extraction layer.
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Convert extracted image features into a 1D vector.
    layers.Flatten(),

    # Combine extracted features.
    layers.Dense(64, activation='relu'),

    # Output one probability for binary classification.
    layers.Dense(1, activation='sigmoid')
])

# Show model structure for public readers and students.
model.summary()

## 4. Compile the Model

Compiling prepares the model for training.  
The model needs to know how to measure errors and how to update itself.

In [ ]:
model.compile(
    optimizer='adam',            # Updates the model weights during training.
    loss='binary_crossentropy',  # Appropriate loss function for Normal vs Pneumonia.
    metrics=['accuracy']         # Reports the percentage of correct predictions.
)

## 5. Convert Dataset to NumPy Arrays

Keras models train on numerical arrays.  
This step converts each image into a normalized array.

Why divide by 255?

- Image pixels usually range from 0 to 255.
- Dividing by 255 changes the range to 0 to 1.
- Neural networks usually train more smoothly with normalized values.

In [ ]:
# Convert all training images to NumPy arrays and normalize pixel values to [0, 1].
x_train = np.array([
    np.array(img, dtype=np.float32) / 255.0
    for img, label in train_dataset
])

# Extract the label for each training image.
# Each label is stored as an array, so label[0] gets the actual number.
y_train = np.array([
    label[0]
    for img, label in train_dataset
])

# Add a channel dimension so each image shape becomes (28, 28, 1).
# CNNs expect image data in the format: height, width, channels.
x_train = x_train[..., np.newaxis]

# Print shapes to confirm the data is ready for training.
print("x_train shape:", x_train.shape)
print("y_train shape:", y_train.shape)

## 6. Train the Model

Training means the model repeatedly looks at the images, makes predictions, compares them to the correct labels, and updates its internal weights.

Key terms:

- **Epoch:** one full pass through the training dataset
- **Batch size:** number of images processed before the model updates its weights
- **Loss:** how wrong the model is
- **Accuracy:** how often the model predicts correctly

In [ ]:
# Train the CNN model using the training images and labels.
history = model.fit(
    x_train, y_train,    # Input images and correct labels.
    epochs=30,           # Train for 30 full passes through the dataset.
    batch_size=64        # Process 64 images at a time before updating weights.
)

## 7. Visualize Training Loss

Loss should generally decrease as the model learns.  
A decreasing loss curve suggests that the model is improving on the training data.

In [ ]:
# Create a new figure for the loss graph.
plt.figure(figsize=(6, 4))

# Plot the training loss recorded after each epoch.
plt.plot(history.history['loss'], label='train loss')

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.legend()
plt.show()

## 8. Visualize Training Accuracy

Accuracy should generally increase as the model learns.  
However, high training accuracy alone does not prove the model works well on unseen data.

In [ ]:
# Create a graph of training accuracy across epochs.
plt.figure(figsize=(6, 4))
plt.plot(history.history['accuracy'], label='train accuracy')

plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training Accuracy')
plt.legend()
plt.show()

## 9. Save the Model

Saving the model stores both:

- the model architecture
- the learned weights

This allows Week 4 to load the trained model and run inference without rebuilding it from scratch.

In [ ]:
# Save the trained model to a file named "my_model.keras".
model.save('my_model.keras')

# Print a message to confirm that the model has been saved.
print("Model saved!")

## Week 3 Summary

By the end of Week 3, students learned how to:

- Prepare image data for CNN training
- Normalize pixel values
- Train a model using `model.fit()`
- Interpret loss and accuracy
- Save a trained Keras model